# Algorithmic Hiring - Curso Introductorio

## Práctica Parte 1: Learning to rank

La primera sesión práctica del curso introductorio al Algorithmic Hiring tiene como objetivo **implementar** en Python un **modelo point-wise learning to rank (LTR)** de ordenamiento de candidatos aplicando técnicas de aprendizaje supervisado.

**Escenario ficticio de práctica**

*Imagina que trabajas para una empresa de recursos humanos como científico de datos y te encomiendan desarrollar una herramienta para el ordenamiento de candidatos. Para ello te proveen datos sobre candidatos que postularon en el pasado a puestos vacantes en áreas como asistente administrativo, gestor/a de proyectos o consultor/a. Tu tarea es usar aprendizaje supervisado para desarrollar un modelo LTR para ordenar candidatos.*

**Datasets**

1. Descargar los datasets para realizar la práctica de [aquí](https://drive.google.com/file/d/16e9EVfscqG03cRV3le0du6TZoayePvr-/view?usp=drive_link).
2. Descomprimir localmente el archivo descargado (`.zip`)
3. Subir cada fichero al directorio raíz de Google Drive.

*Los datasets fueron creados por investigadores del proyecto [FINDHR](https://findhr.eu) y contienen datos ficticios para ser usados con fines académicos.*

> **Completar estos datos antes de entregar**

Nombre: <font color="blue">Pol Mazón Caballero</font>

Email: <font color="blue">pol.mazon@gmail.com</font>

Fecha: <font color="blue">31-07-2026</font>

In [ ]:
# [NO MODIFICAR] carga librerias requeridas
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd

from ast import literal_eval
from datetime import datetime
from google.colab import drive
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import ndcg_score, r2_score
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit

In [ ]:
# [NO MODIFICAR] otorga permisos al notebook para importar datos del drive
drive.mount('/content/drive')

## 1. Importar y explorar datos

In [ ]:
# [NO MODIFICAR] indica ruta a la carpeta donde se encuentra el dataset
data_dir = '/content/drive/MyDrive/'

In [ ]:
# [NO MODIFICAR] carga datos de candidatos
candidates_df = pd.read_csv(
    os.path.join(data_dir, 'candidates_mudab.csv'),
    converters={
        'professional_experience_c': literal_eval,
        'education_background_c': literal_eval,
        'skills_c': literal_eval
    }
)

In [ ]:
# [NO MODIFICAR] carga datos de vacancias
jobs_df = pd.read_csv(
    os.path.join(data_dir, 'jobs_mudab.csv'),
    converters={
        'experience_reqs_role_j': literal_eval,
        'education_reqs_j': literal_eval,
        'skills_j': literal_eval
    }
)

In [ ]:
# [NO MODIFICAR] carga datos de aplicaciones
applications_df = pd.read_csv(os.path.join(data_dir, 'applications_mudab.csv'))

### Candidatos

In [ ]:
# [NO MODIFICAR] muestra la cantidad de registros en el dataset de candidatos
print(f'El dataset de incluye datos de {candidates_df.shape[0]} candidatos')

In [ ]:
# [NO MODIFICAR] visualiza los primeros registros del dataset de candidatos
candidates_df.head()

> **Diccionario variables (`candidatos`)**
>
>
> * `id_c`: identificador (entero)
> * `education_background_c`: formación educativa (lista de diccionarios en el que cada diccionario contiene la siguiente información: `institution`, `start_date`, `end_date`, `degree`)
> * `professional_experience_c`: experiencia profesional (lista de diccionarios, cada uno con: `institution`, `start_date`, `end_date`, `role`, `duration`, `duration_months`)
> * `skills_c`: habilidades (lista de textos)
> * `gender_c`: género (Man/Woman)
> * `origin_c`: procedencia (EU/Non-EU)

### Vacantes

In [ ]:
# [NO MODIFICAR] muestra la cantidad de registros en el dataset de vacancias
print(f'El dataset tiene vacancias para {jobs_df.shape[0]} puestos de trabajo')

In [ ]:
# [NO MODIFICAR] visualiza los primeros registros del dataset de vacancias
jobs_df.head()

> **Diccionario variables (`vacantes`)**
>
>
> * `id_j`: identificador (entero)
> * `education_reqs_j`: requisitos educativos (lista de textos)
> * `experience_reqs_role_j`: requisitos profesionales (lista de textos)
> * `experience_reqs_duration_j`: meses requeridos en puesto similar (entero)
> * `skills_j`: requisitos de habilidades (lista de textos)

### Aplicaciones

In [ ]:
# [NO MODIFICAR] muestra la cantidad de registros en el dataset de aplicaciones
print(f'El dataset está compuesto por {applications_df.shape[0]} aplicaciones')

In [ ]:
# [NO MODIFICAR] visualiza los primeros registros del dataset de aplicaciones
applications_df.head()

> **Diccionario variables (`aplicaciones`)**
>
>
> * `id_c`: identificador de candidato (entero)
> * `id_j`: identificador de vacante (entero)
> * `ranking`: ranking del candidato `c` en vacante `j` (entero)
> * `score`: encaje del candidato `c` en vacante `j` (decimal entre 0 y 1)

## 2. Combinar datasets

Lo primero es combinar los datasets en un solo dataframe utilizando los identificadores `id_c` y `id_j` como claves. Para ello utilizamos la función `merge` de pandas como se muestra a continuación.

```python
dataframe_A.merge(dataframe_B, left_on='key_A', right_on='key_B')
```

In [ ]:
# [NO MODIFICAR] combina los datasets de candidatos y aplicaciones
data_df = candidates_df.merge(applications_df, left_on='id_c', right_on='id_c')

<font color='red'>Usar la celda de abajo para combinar el dataframe `data_df` recién creado con el de vacantes guardando el resultado en `data_df`.</font>

In [ ]:
# combina el dataset data_df con el de vacantes usando id_j como clave
data_df = data_df.merge(jobs_df, left_on='id_j', right_on='id_j')

In [ ]:
# [NO MODIFICAR] visualiza el dataset resultante
data_df.head()

In [ ]:
# [NO MODIFICAR] visualiza la cantidad de registros en el dataset resultante
print(f'El dataset combinado tiene {data_df.shape[0]} registros')

## 3. Dividir el dataset en datos de entrenamiento y prueba

Lo siguiente es separar una porción del dataset, reservándola para cuando toque evaluar el modelo. Para ello dividimos el dataset en dos subconjuntos: uno para ser utilizado en el entrenamiento (80%) y otro para la evaluación (20%), utilizando la función

```python
train_test_split(X, y, test_size=0.20, random_state=42)
```

Por intuición y también examinando una muestra de los datos, se nota que el encaje (`score`) aparenta ser muy importante para determinar el ranking de un candidato. Por este motivo, debemos asegurarnos de que tanto el conjunto de entrenamiento como el de prueba sean representativos, conteniendo valores para los diferentes tipos de encajes. Esto lo hacemos agrupando los valores de `score` y luego aplicando una división estratificada de los datos.

In [ ]:
# [NO MODIFICAR] asigna los valores de score a 5 grupos de la siguiente manera
# grupo 1: 0.0 >= score < 0.2
# grupo 2: 0.2 >= score < 0.4
# grupo 3: 0.4 >= score < 0.6
# grupo 4: 0.6 >= score < 0.8
# grupo 5: 0.8 >= score
data_df['score_cat'] = pd.cut(
    data_df['score'],
    bins=[0.0, 0.2, 0.4, 0.6, 0.8, 1],
    labels=[1, 2, 3, 4, 5],
    include_lowest=True
)

In [ ]:
# [NO MODIFICAR] separa aplicando division estratificada 20% del dataset para ser
# utilizado en la evaluación del modelo
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in split.split(data_df, data_df['score_cat']):
  data_df_test = data_df.loc[test_idx]
  data_df = data_df.loc[train_idx]
print(f'El dataset de entrenamiento quedo con {data_df.shape[0]} registros')
print(f'El dataset para prueba tiene {data_df_test.shape[0]} registros')

In [ ]:
# [NO MODIFICAR] elimina la variable score_cat de los datasets, su proposito ya
# esta cumplido
data_df = data_df.drop(columns=['score_cat'])
data_df_test = data_df_test.drop(columns=['score_cat'])

## 3. Visualizar datos

### 3.1 Visualizar distribución de género

In [ ]:
# [NO MODIFICAR] visualiza la distribución de género entre los candidatos del dataset
ax = data_df['gender_c'].value_counts().plot(
    kind='bar',
    xlabel='Género',
    ylabel='Frecuencia',
    rot=0,
    color=['green','purple']
)
for p in ax.patches:
    b = p.get_bbox()
    ax.annotate(str(p.get_height()), ((b.x0 + b.x1)/2, b.y1+7))
plt.show()

### 3.2 Visualizar distribución de procedencia

<font color="red">Utilizar la celda de abajo para mostrar la distribución de procedencias de los candidatos del dataset. Puedes seguir como ejemplo el código usado para visualizar la distribución de género. En particular, la siguiente función puede ser útil.</font>

```python
data_df['variable'].value_count().plot(kind='bar', xlabel='Nombre variable', ylabel='Frecuencia', rot=0, color=['color 1','color 2'])

```

In [ ]:
# dibuja la distribucion de procedencia entre los candidatos del dataset
ax = data_df['origin_c'].value_counts().plot(
    kind='bar',
    xlabel='Procedencia',
    ylabel='Frecuencia',
    rot=0,
    color=['orange','blue']
)
for p in ax.patches:
    b = p.get_bbox()
    ax.annotate(str(p.get_height()), ((b.x0 + b.x1)/2, b.y1+7))
plt.show()

<font size="+0.5" color="red">🤔<b>¿Por qué el total de Europeos y No Europeos (1144) supera la cantidad total de candidatos (861)?</b></font>

Porque `data_df` es el dataset combinado a nivel de **aplicación**, es decir, contiene una fila por cada aplicación de un candidato a una vacante y no una fila por candidato único. Un mismo candidato puede haber aplicado a más de una vacante, por lo que aparece repetido varias veces en `data_df` (una vez por cada aplicación que realizó). Al contar los valores de `origin_c` sobre este dataset, cada aplicación del candidato se cuenta como un registro independiente, por lo que el total de conteos (1144) supera el número de candidatos únicos (861) presentes en `candidates_df`.

### 3.3 Visualizar la distribución de encaje

In [ ]:
# [NO MODIFICAR] visualiza la distribución de los puntajes de encaje
data_df['score'].plot(kind='hist', xlabel='Puntaje de encaje', ylabel='Frecuencia')
plt.show()

## 4. Procesar dataset

### 4.1 Explorar completitud de variables

Antes del entrenamiento es importante verificar que las variables del dataset estén completas. Verifiquemos la variable género (`gender_c`) para comprobar si tiene valores en todos los registros.

In [ ]:
# [NO MODIFICAR] muestra la cantidad de valores faltantes en la variable género
num_missing_values = data_df[data_df['gender_c'].isna()].shape[0]
print(f'A la variable genero le faltan {num_missing_values} valores')

<font color="red">Comprueba, utilizando la celda de abajo, la completitud de la variable procedencia (`origin_c`)</font>

In [ ]:
# verifica valores faltantes en la variable procedencia
num_missing_values = data_df[data_df['origin_c'].isna()].shape[0]
print(f'A la variable procedencia le faltan {num_missing_values} valores')

<font color="red">Comprueba, utilizando la celda de abajo, la completitud de la variable encaje (`score`)</font>

In [ ]:
# verifica valores faltantes en la variable encaje
num_missing_values = data_df[data_df['score'].isna()].shape[0]
print(f'A la variable encaje le faltan {num_missing_values} valores')

La variable **`ranking`** debería estar completa ya que para utilizar aprendizaje supervisado se requiere que cada registro tenga su correspondiente "etiqueta", en este caso el ranking. Verifiquemos que sea así.

In [ ]:
# [NO MODIFICAR] muestra la cantidad de valores faltantes en la variable ranking
num_missing_values = data_df[data_df['ranking'].isna()].shape[0]
print(f'A la variable ranking le faltan {num_missing_values} valores')

### 4.2 Procesar variables categóricas

Los algoritmos de aprendizaje supervisado requieren que todas las variables de entrada sean numéricas. En nuestro caso, tenemos dos variables categóricas: género (`gender_c`) y procedencia (`origin_c`), que necesitan ser convertidas a enteros antes de ser introducidas al algoritmo.

<font color="red">Utiliza la celda de abajo para convertir la variable `gender_c`, asignando `1` al valor `Woman` mientras que `0` al valor `Man`. La siguiente función de la librería `numpy (np)` puede resultar útil, aunque existen otras maneras de hacerlo.</font>

```python
np.where([condicion],[valor si condicion es verdadera],[valor si condicion es falsa])
```

In [ ]:
# convierte la variable gender_c a numerica (1 = Woman, 0 = Man)
data_df['gender_c'] = np.where(data_df['gender_c'] == 'Woman', 1, 0)

<font color='red'>A continuación se convierte la variable `origin_C` a numérica usando el `1` para los valores `Non-EU` mientras que `0` para el resto.</font>

In [ ]:
# convierte la variable procedencia a numerica (1 = Non-EU, 0 = EU)
data_df['origin_c'] = np.where(data_df['origin_c'] == 'Non-EU', 1, 0)

## 4.3 Crear nuevas variables (features)

Además de las variables de género y procedencia es necesario crear otras que representen de forma numérica las características profesionales y educativas de los candidatos. A este proceso se le conoce como feature engineering.

### 4.3.1 Experiencia profesional acumulada

La primera variable que crearemos representará el total de experiencia de los candidatos en meses. Para ello iteraremos sobre la lista de experiencias profesionales sumando la duración (en meses) de cada una. El resultado se alojará en la variable `exp_months`.

In [ ]:
# [NO MODIFICAR] calcula los meses de experiencia profesional de los candidatos
def exp_months(data_df):
  data_df['exp_months'] = 0
  for idx, row in data_df.iterrows():
    candidate_exp = row['professional_experience_c']
    candidate_id = row['id_c']
    total_exp = 0
    for exp in candidate_exp:
      total_exp += int(exp['duration_months'])
    data_df.loc[data_df['id_c']==candidate_id, 'exp_months'] = total_exp
  return data_df

data_df = exp_months(data_df)

### 4.3.2 Cumple requisito de experiencia profesional

Todas las vacantes requieren una cantidad mínima de tiempo en puestos similares, por ejemplo para el puesto de consultor se requiere una experiencia de al menos 12 meses en roles parecidos. La siguiente variable será booleana y almacenará verdadero (`1`) o falso (`0`) dependiendo si el candidato cumple o no con este requisito.

<font color='red'>Utiliza la celda de abajo para crear la variable `has_prof_exp` siguiendo las indicaciones de arriba.</font>

In [ ]:
# crea la variable has_prof_exp: 1 si el candidato cumple con el minimo de
# experiencia requerido para el puesto, 0 en caso contrario
def has_prof_exp(data_df):
  data_df['has_prof_exp'] = np.where(
      data_df['exp_months'] >= data_df['experience_reqs_duration_j'], 1, 0
  )
  return data_df

data_df = has_prof_exp(data_df)

### 4.3.3 Ocupó rol en el pasado

Las vacantes requieren que el candidato haya ocupado el rol que se busca en el pasado. La siguiente variable será booleana y almacenará verdadero (`1`) o falso (`0`) dependiendo si el candidato ocupó el rol en el pasado.

<font color='red'>Usa la celda de abajo para crear la variable `occupied_role` siguiendo las instrucciones de arriba.</font>

In [ ]:
# crea la variable occupied_role: 1 si el candidato ocupo en el pasado alguno
# de los roles requeridos por la vacante, 0 en caso contrario
def occupied_role(data_df):
  data_df['occupied_role'] = 0
  for idx, row in data_df.iterrows():
    candidate_roles = [exp['role'].lower() for exp in row['professional_experience_c']]
    required_roles = [r.lower() for r in row['experience_reqs_role_j']]
    occupied = 0
    for role in candidate_roles:
      if role in required_roles:
        occupied = 1
        break
    data_df.loc[idx, 'occupied_role'] = occupied
  return data_df

data_df = occupied_role(data_df)

### 4.3.4 Tiempo de formación académica

A continuación crearemos una variable que representa la cantidad total de meses de formación de los candidatos. Para ello iteraremos sobre las experiencias de formación académica del candidato calculando la duración *solo* de las **experiencias culminadas**, acumulando el total en la variable `edu_months`.

La duración de las experiencias se puede obtener computando la diferencia en meses entre las fechas de conclusión (`end_date`) e inicio (`start_date`) de la formación. Las **experiencias no culminadas** se reconocen porque no tienen una fecha asignada a `end_date` sino el valor `ongoing`. En esos casos se puede asignar la fecha actual como `end_date` para luego realizar el cálculo.

<font color='red'>Utiliza la celda de abajo para calcular la variable `edu_months`. La librería estándar de Python `datetime` puede resultar útil, por ejemplo

```python
end_date = datetime(2025, 12, 20)
start_date = datetime(2025, 10, 22)
diferencia_meses = (end_date.year - start_date.year) * 12 + (end_date.month - start_date.month)
```

In [ ]:
# calcula la variable edu_months: total de meses de formacion academica del
# candidato. Para las formaciones en curso ('ongoing') se usa la fecha actual
# como fecha de fin.
def edu_months(data_df):
  data_df['edu_months'] = 0
  for idx, row in data_df.iterrows():
    candidate_edu = row['education_background_c']
    total_edu = 0
    for edu in candidate_edu:
      start_date = pd.to_datetime(edu['start_date'])
      if edu['end_date'] == 'ongoing':
        end_date = datetime.now()
      else:
        end_date = pd.to_datetime(edu['end_date'])
      total_edu += (end_date.year - start_date.year) * 12 + (end_date.month - start_date.month)
    data_df.loc[idx, 'edu_months'] = total_edu
  return data_df

data_df = edu_months(data_df)

### 4.3.5 Número de habilidades para el puesto

La última variable a calcular será el número de habilidades del candidato que coinciden con las habilidades requeridas para el puesto. Para ello iteraremos sobre la lista de habilidades de los candidatos y la lista de habilidades requeridas para el puesto sumando aquellas en común y el resultado se alojará en la variable `num_job_skills`.

In [ ]:
# [NO MODIFICAR] calcula el numero de habilidades del candidato requeridas para el puesto
def num_job_skills(data_df):
  data_df['num_job_skills'] = 0
  for idx, row in data_df.iterrows():
    candidate_skills = [s.lower() for s in row['skills_c']]
    job_req_skills = [s.lower() for s in row['skills_j']]
    candidate_id = row['id_c']
    num_skills = 0
    for skill in candidate_skills:
      if skill in job_req_skills:
        num_skills += 1
    data_df.loc[data_df['id_c']==candidate_id, 'num_job_skills'] = num_skills
  return data_df

data_df = num_job_skills(data_df)

### 4.3.6 Visualizar nuevas variables

In [ ]:
# [NO MODIFICAR] visualizar nuevas variables
new_variables = ['exp_months', 'has_prof_exp', 'occupied_role', 'edu_months', 'num_job_skills']
data_df[list(candidates_df.columns) + new_variables].head()

## 5. Entrenamiento

Antes del entrenamiento, cargamos en la variable `X_train` los datos correspondientes a las características de los candidatos que nos interesan incluir en el entrenamiento. En este caso, todas las **variables nuevas**, más género (`gender_c`) y procedencia (`origin_c`), así como también el encaje (`score`). Además, cargamos en la variable `y_train` los datos del ranking de los candidatos.

In [ ]:
# [NO MODIFICAR] carga caracteristicas de los candidatos en X y ranking en y
X_train = data_df[new_variables + ['gender_c', 'origin_c', 'score']]
y_train = data_df['ranking']

Utilizando los conjuntos de datos `X_train` e `y_train` procederemos a entrenar un modelo de learning to rank (LTR) usando el algoritmo de regresión de `random forest`.

 <font color="red">Utiliza la celda de abajo para entrenar un modelo de LTR con la siguiente función</font>

 ```python
 model = RandomForestRegressor().fit(x, y)
 ```

In [ ]:
# entrena un modelo de regresion random forest
model = RandomForestRegressor(random_state=42).fit(X_train, y_train)

## 6. Evaluación

La evaluación de modelos LTR se realiza principalmente por medio de métricas que ayudan a entender qué tan bueno es el modelo ordenando items de una lista. Una de estas métricas es el *[normalized discounted cumulative gain (NDCG)](https://medium.com/data-science/demystifying-ndcg-bee3be58cfe0)* que verifica que tanto los mejores candidatos de la lista se encuentren principalmente en los primeros lugares de la clasificación. El resultado es un número entre 0 y 1 que cuanto más cercano a 1 mejor.

En menor medida el modelo puede ser evaluado usando métricas convencionales de machine learning, como por ejemplo el $r^2$, la cual mide qué tan bien el modelo predice el valor de la variable objetivo (`ranking`). Al igual que con *NCDG*, el resultado es un número entre 0 y 1 que cuanto más cercano a 1 mejor.

<font color='red'>Utiliza las celdas siguientes para evaluar el modelo aplicando las métricas *NDCG* y *r^2* al resultado de las predicciones sobre el conjunto de prueba (`data_df_test`).</font>

```python
ndcg = ndcg_score([y_test], [predictions])
print(f'La calidad del ranking producida por el modelo es: {ndcg}')
```

```python
r2 = r2_score(y_test, y_pred)
print(f'La capacidad del modelo para predecir los valores del ranking es: {r2}')
```

<font color="red">❗Antes de realizar la evaluación debes preparar el conjunto de pruebas utilizando los mismos métodos aplicados al conjunto de entrenamiento. Esto significa convertir valores categóricos a numéricos, crear variables nuevas, y luego separar las características de los candidatos de su ranking, tal como lo hicimos anteriormente.</font>

In [ ]:
# convierte variables categoricas a numericas en el conjunto de prueba
data_df_test['gender_c'] = np.where(data_df_test['gender_c'] == 'Woman', 1, 0)
data_df_test['origin_c'] = np.where(data_df_test['origin_c'] == 'Non-EU', 1, 0)

In [ ]:
# crea la variable exp_months en el conjunto de prueba
data_df_test = exp_months(data_df_test)

In [ ]:
# crea la variable has_prof_exp en el conjunto de prueba
data_df_test = has_prof_exp(data_df_test)

In [ ]:
# crea la variable occupied_role en el conjunto de prueba
data_df_test = occupied_role(data_df_test)

In [ ]:
# crea la variable edu_months en el conjunto de prueba
data_df_test = edu_months(data_df_test)

In [ ]:
# crea la variable num_job_skills en el conjunto de prueba
data_df_test = num_job_skills(data_df_test)

In [ ]:
# [NO MODIFICAR] Separa las caracteristicas de los candidatos de su ranking
X_test = data_df_test[new_variables + ['gender_c', 'origin_c', 'score']]
y_test = data_df_test['ranking']

In [ ]:
# [NO MODIFICAR] Estima el ranking de los candidatos en el conjunto de prueba
predictions = model.predict(X_test)

In [ ]:
# evalua el modelo usando la metrica NDCG
ndcg = ndcg_score([y_test], [predictions])
print(f'La calidad del ranking producida por el modelo es: {ndcg}')

In [ ]:
# evalua la capacidad del modelo para predecir los valores del ranking
r2 = r2_score(y_test, predictions)
print(f'La capacidad del modelo para predecir los valores del ranking es: {r2}')

<font color="red"><b>🤔 ¿Qué significa este resultado?</b></font>

Un valor de $r^2$ relativamente bajo indica que el modelo no predice con exactitud el valor numérico del `ranking` de cada candidato. Sin embargo, esto **no invalida** el modelo como herramienta de LTR. El objetivo de un modelo point-wise de learning to rank no es acertar el valor exacto de la posición de cada candidato, sino producir un **orden relativo** correcto entre los candidatos: que los mejores candidatos aparezcan antes que los peores en la lista final. Esa capacidad se mide con *NDCG*, que sí puede ser alta incluso cuando el $r^2$ es bajo, porque penaliza únicamente los errores de orden (y con mayor peso los que ocurren en las primeras posiciones), no la distancia numérica exacta entre el ranking predicho y el real. Por lo tanto, para decidir si el modelo es válido para esta tarea hay que fijarse principalmente en la métrica *NDCG* y no en el $r^2$.

## 7. Aplicación

Como último paso utiliza el modelo creado para ordenar una lista de más de 90 candidatos que han aplicado a un puesto de venta. Los datos, que se pueden descargar de [aquí](https://drive.google.com/file/d/1MFdfTMNASvxakOVp0-Nw-mgOOJAfqt5d/view?usp=drive_link) y proveen la trayectoria educativa y profesional del candidato así como sus habilidades, género y procedencia. Además, el dataset incluye los requerimientos del puesto y el encaje del candidato en el mismo. Sigue las instrucciones al inicio del notebook para descargar y organizar el dataset.

In [ ]:
# [NO MODIFICAR] carga datos de candidatos para vacante de ventas
sales_candidates_df = pd.read_csv(
    os.path.join(data_dir, 'sales_candidates.csv'),
    converters={
        'professional_experience_c': literal_eval,
        'education_background_c': literal_eval,
        'skills_c': literal_eval
    }
)

In [ ]:
# [NO MODIFICAR] carga requerimientos para vacante de ventas
sales_jobs_df = pd.read_csv(
    os.path.join(data_dir, 'sales_job_description.csv'),
    converters={
        'experience_reqs_role_j': literal_eval,
        'education_reqs_j': literal_eval,
        'skills_j': literal_eval
    }
)

In [ ]:
# [NO MODIFICAR] carga datos de aplicaciones para vacante de ventas
applications_df = pd.read_csv(os.path.join(data_dir, 'sales_applications.csv'))

In [ ]:
# [NO MODIFICAR] combina datasets
new_data_df = sales_candidates_df.merge(applications_df, left_on='id_c', right_on='id_c')
new_data_df = new_data_df.merge(sales_jobs_df, left_on='id_j', right_on='id_j')

In [ ]:
# [NO MODIFICAR] visualiza las primeras 5 filas del dataset
new_data_df.head(5)

In [ ]:
# verifica registros incompletos y, de existir, elimina los registros
# incompletos si son menos del 10% o si no aplica una tecnica de imputacion
missing_pct = new_data_df.isna().any(axis=1).mean() * 100
print(f'Porcentaje de registros incompletos: {missing_pct:.2f}%')

if missing_pct < 10:
  new_data_df = new_data_df.dropna().reset_index(drop=True)
  print(f'Se eliminaron los registros incompletos. Quedan {new_data_df.shape[0]} registros')
else:
  # imputacion simple: variables numericas con la mediana, categoricas con la moda
  for col in new_data_df.columns:
    if new_data_df[col].isna().any():
      if pd.api.types.is_numeric_dtype(new_data_df[col]):
        new_data_df[col] = new_data_df[col].fillna(new_data_df[col].median())
      else:
        new_data_df[col] = new_data_df[col].fillna(new_data_df[col].mode()[0])
  print('Se imputaron los valores faltantes')

In [ ]:
# convierte variables categoricas a numericas
new_data_df['gender_c'] = np.where(new_data_df['gender_c'] == 'Woman', 1, 0)
new_data_df['origin_c'] = np.where(new_data_df['origin_c'] == 'Non-EU', 1, 0)

In [ ]:
# crea la variable exp_months
new_data_df = exp_months(new_data_df)

In [ ]:
# crea la variable has_prof_exp
new_data_df = has_prof_exp(new_data_df)

In [ ]:
# crea la variable occupied_role
new_data_df = occupied_role(new_data_df)

In [ ]:
# crea la variable edu_months
new_data_df = edu_months(new_data_df)

In [ ]:
# crea la variable num_job_skills
new_data_df = num_job_skills(new_data_df)

In [ ]:
# ordena los candidatos usando el modelo entrenado, almacenando el resultado
# en la variable pred_orders
X_new = new_data_df[new_variables + ['gender_c', 'origin_c', 'score']]
pred_orders = model.predict(X_new)

In [ ]:
# [NO MODIFICAR] combina los ordenes estimados con los datos de los candidatos y ordena la lista
ordered_data_df = pd.concat([new_data_df, pd.DataFrame({'order': pred_orders})], axis=1)
ordered_data_df = ordered_data_df.sort_values('order', ascending=True)

In [ ]:
# [NO MODIFICAR] elimina la columna order ya que solo la necesitabamos para ordenar
# la lista
ordered_data_df = ordered_data_df.drop(columns=['order'])

In [ ]:
# [NO MODIFICAR] visualiza la lista ordenada de candidatos
ordered_data_df['gender_c'] = np.where(ordered_data_df['gender_c']==1, 'F', 'M')
ordered_data_df['origin_c'] = np.where(ordered_data_df['origin_c']==1, 'Non-EU', 'EU')
ordered_data_df = ordered_data_df.reset_index(drop=True)
ordered_data_df[new_variables + ['gender_c', 'origin_c', 'score']].head(10)

In [ ]:
# [NO MODIFICAR] guarda la lista ordenada de candidatos en archivo CSV para
# utilizarla en la practica 2
ordered_data_df[new_variables + ['gender_c', 'origin_c', 'score']].to_csv(
    os.path.join(data_dir, 'sales_ordered_candidates.csv'), index=False
)

<font color="red"><b>🤔 ¿Notas algún aspecto llamativo en la lista?</b></font>

Al observar los 10 primeros candidatos de la lista ordenada conviene revisar si existe algún patrón repetido en variables sensibles como género (`gender_c`) u procedencia (`origin_c`). Si, por ejemplo, la mayoría de los primeros puestos están ocupados por candidatos de un mismo género o procedencia, incluso cuando otras variables (experiencia, formación, habilidades) son similares entre candidatos de distintos grupos, esto podría ser indicio de que el modelo está reproduciendo patrones o sesgos presentes en los datos históricos de contratación (por ejemplo, si en el pasado se contrató mayoritariamente a un perfil determinado). Esto ilustra un riesgo importante de los sistemas de *algorithmic hiring*: un modelo entrenado sobre decisiones pasadas puede perpetuar o amplificar sesgos existentes, aun cuando las variables de género y procedencia no sean, en teoría, las que determinan directamente el ranking. Por ello es importante auditar los resultados del modelo en busca de estos patrones antes de utilizarlo en un proceso de selección real.

## Entrega (individual)

Para entregar la práctica, seguir las siguientes instrucciones:

*   Descargar el notebook `File->Download->Download .ipynb`
*   Renombrar a `nombre.apellido-practica1.ipynb`
*   Enviar a través del `ECampus` a más tardar el <font color="red"><b>28-05-2026 23:59</b></font>.

<font size="+2" color="blue">Declaro que, excepto el código provisto por el instructor del curso, todo el resto del código, texto y figuras fueron producidos por mí mismo.</font>